In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import sys
import importlib.util
import subprocess
import copy
import gc
import hashlib
import json
import math
import random
import time
import zipfile
from pathlib import Path
try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from scipy.special import expit
from scipy.stats import binomtest
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss, accuracy_score
import matplotlib.pyplot as plt
from IPython.display import display


In [ ]:
MODE = os.environ.get("NAPS_MODE", "pretrained")
if MODE not in {"pretrained", "recalibrate", "train"}:
    raise ValueError("MODE: pretrained, recalibrate или train")
ROOT_DIR = Path("/content") if IN_COLAB else Path.cwd()
DATA_DIR = Path(os.environ.get("NAPS_DATA_DIR", str(ROOT_DIR)))
WEIGHTS_PATH = Path(os.environ.get("NAPS_WEIGHTS_PATH", str(ROOT_DIR / "NAPS_Calo_ResCNN_weights.pt")))
OUTPUT_DIR = Path(os.environ.get("NAPS_OUTPUT_DIR", str(ROOT_DIR / "naps_rescnn")))
CACHE_DIR = Path(os.environ.get("NAPS_CACHE_DIR", str(ROOT_DIR / "naps_rescnn_cache")))
SESSION_PATTERN = "session*.npz"
DOWNLOAD_RESULTS = False
SEED = 20261109
ALPHAS = np.array([0.05, 0.10, 0.20], dtype=np.float64)
RECALIBRATION_GAMMA = 0.0
MAX_BUDGET = 1_000_000
MAX_EPOCHS = int(os.environ.get("NAPS_MAX_EPOCHS", "40"))
PATIENCE = 10
GUARD_DELTA = 0.005
AUDIT_MIN_N = 50
AUDIT_SIGNIFICANCE = 0.05
W_ITERATIONS = int(os.environ.get("NAPS_W_ITERATIONS", "800"))
BORDER_COUNT = 32
E_EDGES = np.geomspace(1.0, 100.0, 7)
G_EDGES = np.linspace(0.7, 1.3, 5)
N_BINS = 24
MARGINS = np.array([0, .125, .25, .5, .75, 1, 1.5, 2, 3, 4, 6, 10, np.inf])
THREADS = min(2, os.cpu_count() or 1)
torch.set_num_threads(THREADS)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
for folder in (OUTPUT_DIR, CACHE_DIR, OUTPUT_DIR / "figures"):
    folder.mkdir(parents=True, exist_ok=True)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True)

seed_everything(SEED)
print({"model": "ResCNN", "mode": MODE, "device": str(DEVICE)})


In [ ]:
COORD = np.linspace(-1, 1, 8, dtype=np.float32)
GRID_X, GRID_Y = np.meshgrid(COORD, COORD)

def validate_observations(raw):
    x = np.asarray(raw, dtype=np.float32)
    if x.ndim != 4 or x.shape[1:] != (4, 8, 8):
        raise ValueError("X должен иметь форму (N,4,8,8)")
    if not np.isfinite(x).all() or np.any(x < 0):
        raise ValueError("X должен содержать конечные неотрицательные энергии")
    return x

def event_features(raw):
    x = validate_observations(raw)
    layer = x.sum(axis=(2, 3))
    total = layer.sum(axis=1)
    denominator = np.maximum(total, 1e-8)
    fractions = layer / denominator[:, None]
    image = x.sum(axis=1)
    cx = (image * GRID_X).sum(axis=(1, 2)) / denominator
    cy = (image * GRID_Y).sum(axis=(1, 2)) / denominator
    wx = np.sqrt(np.maximum((image * GRID_X**2).sum(axis=(1, 2)) / denominator - cx**2, 0))
    wy = np.sqrt(np.maximum((image * GRID_Y**2).sum(axis=(1, 2)) / denominator - cy**2, 0))
    z = np.linspace(0, 1, 4, dtype=np.float32)
    cz = (fractions * z).sum(axis=1)
    wz = np.sqrt(np.maximum((fractions * z**2).sum(axis=1) - cz**2, 0))
    return np.column_stack([
        np.log1p(total), np.log1p(layer), fractions,
        (x > 0).mean(axis=(2, 3)), np.log1p(x.max(axis=(1, 2, 3))),
        cx, cy, wx, wy, cz, wz
    ]).astype(np.float32)

def numpy_scaler(preprocessing):
    return {key: value.detach().cpu().numpy() if isinstance(value, torch.Tensor)
            else np.asarray(value) for key, value in preprocessing.items()}

def preprocess(raw, scaler, features=None):
    x = validate_observations(raw)
    s = numpy_scaler(scaler)
    logmaps = (np.log1p(x) - s["image_mean"][None, :, None, None]) / s["image_std"][None, :, None, None]
    relative = x / np.maximum(x.sum(axis=(1, 2, 3)), 1e-8)[:, None, None, None]
    maps = np.concatenate([logmaps, (x > 0).astype(np.float32), relative], axis=1)
    f = event_features(x) if features is None else np.asarray(features, dtype=np.float32)
    f = (f - s["feature_mean"]) / s["feature_std"]
    return maps.astype(np.float32), f.astype(np.float32)

class ResidualBlock(nn.Module):
    def __init__(self, width, dropout):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(width, width, 3, padding=1, bias=False), nn.GroupNorm(4, width), nn.SiLU(),
            nn.Dropout2d(dropout), nn.Conv2d(width, width, 3, padding=1, bias=False), nn.GroupNorm(4, width)
        )

    def forward(self, x):
        return nn.functional.silu(x + self.layers(x))

class CaloResCNN(nn.Module):
    def __init__(self, config):
        super().__init__()
        if (config["arch"] != "rescnn" or not config["use_shape"] or not config["use_features"]):
            raise ValueError("Этот Colab поддерживает только выбранную ResCNN")
        width, dropout = config["width"], config["dropout"]
        self.encoder = nn.Sequential(
            nn.Conv2d(12, width, 3, padding=1), nn.SiLU(),
            *[ResidualBlock(width, dropout) for _ in range(config["blocks"])],
            nn.AdaptiveAvgPool2d((4, 4)), nn.Flatten()
        )
        self.feature_encoder = nn.Sequential(nn.Linear(20, 32), nn.SiLU(), nn.Linear(32, 32), nn.SiLU())
        self.shared = nn.Sequential(nn.Linear(width * 16 + 32, 128), nn.SiLU(), nn.Dropout(dropout))
        self.class_head = nn.Linear(128, 1)
        self.nuisance_head = nn.Linear(128, 8)

    def forward(self, x, features):
        h = torch.cat([self.encoder(x), self.feature_encoder(features)], dim=1)
        h = self.shared(h)
        nuisance = self.nuisance_head(h).reshape(-1, 2, 4)
        mu = torch.sigmoid(nuisance[:, :, :2])
        scale = .025 + nn.functional.softplus(nuisance[:, :, 2:])
        return self.class_head(h).squeeze(1), mu, scale

def statistics(logits, train_prior_1):
    if not 0 < train_prior_1 < 1:
        raise ValueError("Обучающая доля класса 1 должна лежать между 0 и 1")
    prior_logit = math.log(train_prior_1 / (1 - train_prior_1))
    log_bf = np.asarray(logits, dtype=np.float64) - prior_logit
    return np.column_stack([-log_bf, log_bf])

def encode_membership(membership):
    m = np.asarray(membership, dtype=bool)
    return m[..., 0].astype(np.uint8) + 2 * m[..., 1].astype(np.uint8)


In [ ]:
if not WEIGHTS_PATH.is_file() and IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    matching = [Path(name) for name in uploaded if name.endswith(".pt")]
    if len(matching) == 1:
        WEIGHTS_PATH = matching[0]
if not WEIGHTS_PATH.is_file():
    raise FileNotFoundError(f"Загрузите NAPS_Calo_ResCNN_weights.pt и укажите WEIGHTS_PATH: {WEIGHTS_PATH}")
SOURCE_CHECKPOINT = torch.load(WEIGHTS_PATH, map_location="cpu", weights_only=True)
if SOURCE_CHECKPOINT.get("format_version") != 1:
    raise ValueError("Нужен поставляемый checkpoint NAPS_Calo_ResCNN_weights.pt")
CONFIG = copy.deepcopy(SOURCE_CHECKPOINT["config"])
CHECKPOINT = copy.deepcopy(SOURCE_CHECKPOINT)
MODEL = CaloResCNN(CONFIG).to(DEVICE)
MODEL.load_state_dict(CHECKPOINT["state_dict"], strict=True)
MODEL.eval()
SCALER = CHECKPOINT["preprocess"]
TRAIN_PRIOR_1 = float(CHECKPOINT["train_prior_1"])
display(pd.DataFrame([{
    "model": CHECKPOINT["model_name"], "width": CONFIG["width"], "blocks": CONFIG["blocks"],
    "dropout": CONFIG["dropout"], "parameters": sum(p.numel() for p in MODEL.parameters()),
    "saved_gamma": CHECKPOINT["naps"]["gamma"],
    "saved_confidence_set": CHECKPOINT["naps"]["confidence_set"],
    "continuum_certified": CHECKPOINT["naps"]["continuum_certified"],
    "weights_retrained_during_export": CHECKPOINT["provenance"]["weights_retrained_during_export"]
}]))


In [ ]:
@torch.inference_mode()
def predict_logits(model, raw, scaler, batch_size=2048, indices=None):
    if batch_size < 1:
        raise ValueError("batch_size должен быть положительным")
    if np.ndim(raw) != 4 or np.shape(raw)[1:] != (4, 8, 8):
        raise ValueError("X должен иметь форму (N,4,8,8)")
    size = len(raw) if indices is None else len(indices)
    if size == 0:
        return np.empty(0, dtype=np.float64)
    model.eval()
    logits = []
    for start in range(0, size, batch_size):
        batch = raw[start:start + batch_size] if indices is None else raw[np.asarray(indices[start:start + batch_size])]
        maps, features = preprocess(batch, scaler)
        score, _, _ = model(torch.from_numpy(maps).to(DEVICE), torch.from_numpy(features).to(DEVICE))
        logits.append(score.float().cpu().numpy().astype(np.float64))
    return np.concatenate(logits)

class ResCNNNAPSPredictor:
    def __init__(self, checkpoint, device="cpu"):
        checkpoint = torch.load(checkpoint, map_location="cpu", weights_only=True) if isinstance(checkpoint, (str, Path)) else checkpoint
        self.device = torch.device(device)
        self.model = CaloResCNN(checkpoint["config"]).to(self.device)
        self.model.load_state_dict(checkpoint["state_dict"], strict=True)
        self.model.eval()
        self.scaler = numpy_scaler(checkpoint["preprocess"])
        self.prior = float(checkpoint["train_prior_1"])
        self.alphas = checkpoint["naps"]["alphas"].cpu().numpy()
        self.thresholds = checkpoint["naps"]["thresholds"].cpu().numpy()
        if self.thresholds.shape != (len(self.alphas), 2) or np.isnan(self.thresholds).any():
            raise ValueError("Некорректные пороги NAPS")
        if not np.all(self.thresholds[1:] >= self.thresholds[:-1]):
            raise ValueError("Пороги нарушают вложенность prediction sets")

    @torch.inference_mode()
    def predict(self, X, batch_size=2048):
        if batch_size < 1:
            raise ValueError("batch_size должен быть положительным")
        if np.ndim(X) != 4 or np.shape(X)[1:] != (4, 8, 8):
            raise ValueError("X должен иметь форму (N,4,8,8)")
        if len(X) == 0:
            return np.empty(0), np.empty((0, len(self.alphas)), dtype=np.uint8)
        logits = []
        for start in range(0, len(X), batch_size):
            maps, features = preprocess(X[start:start + batch_size], self.scaler)
            l, _, _ = self.model(torch.from_numpy(maps).to(self.device), torch.from_numpy(features).to(self.device))
            logits.append(l.float().cpu().numpy().astype(np.float64))
        logits = np.concatenate(logits)
        membership = statistics(logits, self.prior)[:, None, :] > self.thresholds[None, :, :]
        return expit(logits), encode_membership(membership)

    def export(self, X, path, event_ids=None, batch_size=2048):
        _, codes = self.predict(X, batch_size)
        ids = np.arange(len(X)) if event_ids is None else np.asarray(event_ids)
        if ids.ndim != 1 or len(ids) != len(X):
            raise ValueError("Количество event_id не совпадает с количеством X")
        frame = pd.DataFrame({"event_id": ids})
        for ia, alpha in enumerate(self.alphas):
            frame[f"H_{alpha:.2f}"] = codes[:, ia]
        frame.to_csv(path, index=False)
        return frame


In [ ]:
def file_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(2**20), b""):
            h.update(block)
    return h.hexdigest()

def nuisance_bins(E, g):
    ie = np.clip(np.searchsorted(E_EDGES, E, side="right") - 1, 0, 5)
    ig = np.clip(np.searchsorted(G_EDGES, g, side="right") - 1, 0, 3)
    return ie * 4 + ig

def normalize_nuisance(E, g):
    return np.column_stack([np.log(E) / np.log(100.0), (np.asarray(g) - .7) / .6]).astype(np.float32)

def load_dictionaries(data_dir, expected_manifest=None):
    paths = sorted(Path(data_dir).glob("calo_dictionary*.npz"))
    if not paths:
        return None
    unique = {}
    for path in paths:
        unique.setdefault(file_hash(path), path)
    if expected_manifest is not None:
        expected = [row["sha256"] for row in expected_manifest]
        if set(unique) != set(expected):
            raise ValueError("Для готовых весов нужны исходные NPZ из data_manifest; иные файлы допустимы в MODE='train'")
        ordered = [(digest, unique[digest]) for digest in expected]
    else:
        ordered = list(unique.items())
    specifications, keys, manifest = [], [], []
    for digest, path in ordered:
        with np.load(path, allow_pickle=False) as d:
            if not {"keys", "values"} <= set(d.files):
                raise ValueError(f"Нужны keys и values: {path.name}")
            k = np.asarray(d["keys"], dtype=np.float64)
            v = d["values"]
            if k.ndim != 2 or k.shape[1] != 3 or v.ndim != 5 or len(v) != len(k) or v.shape[2:] != (4, 8, 8):
                raise ValueError(f"Некорректные формы массивов: {path.name}")
            if not np.isfinite(k).all() or not np.isfinite(v).all() or np.any(v < 0):
                raise ValueError(f"Некорректные энергии/параметры: {path.name}")
            if not np.isin(k[:, 0], [0, 1]).all() or np.any((k[:, 1] < 1) | (k[:, 1] > 100) | (k[:, 2] < .7) | (k[:, 2] > 1.3)):
                raise ValueError(f"Параметры вне допустимых диапазонов: {path.name}")
            specifications.append((path, len(k), v.shape[1]))
            keys.append(k)
            manifest.append({"file": path.name, "sha256": digest, "events": int(len(k) * v.shape[1])})
    keys = np.concatenate(keys)
    if len(np.unique(keys, axis=0)) != len(keys):
        raise ValueError("Повторные ключи (y,E,g); независимость событий не установлена")
    conditions, key_groups = np.unique(keys[:, 1:], axis=0, return_inverse=True)
    if not np.all(np.bincount(key_groups) == 2):
        raise ValueError("Нужны оба класса на каждом условии (E,g)")
    n = sum(row["events"] for row in manifest)
    if n > MAX_BUDGET:
        raise ValueError(f"Бюджет превышен: {n} > {MAX_BUDGET}")
    raw = np.lib.format.open_memmap(CACHE_DIR / "raw.npy", mode="w+", dtype="float32", shape=(n, 4, 8, 8))
    labels, energies, gains, groups = [], [], [], []
    event_offset = key_offset = 0
    for path, n_keys, repeats in specifications:
        k = keys[key_offset:key_offset + n_keys]
        with np.load(path, allow_pickle=False) as d:
            raw[event_offset:event_offset + n_keys * repeats] = d["values"].reshape(-1, 4, 8, 8)
        labels.append(np.repeat(k[:, 0].astype(np.int64), repeats))
        energies.append(np.repeat(k[:, 1], repeats))
        gains.append(np.repeat(k[:, 2], repeats))
        groups.append(np.repeat(key_groups[key_offset:key_offset + n_keys], repeats))
        event_offset += n_keys * repeats
        key_offset += n_keys
    raw.flush()
    y, E, g, group = map(np.concatenate, (labels, energies, gains, groups))
    counts = np.bincount(2 * group + y, minlength=2 * len(conditions)).reshape(-1, 2)
    if not np.all(counts[:, 0] == counts[:, 1]):
        raise ValueError("Нужны одинаковые числа повторов обоих классов на условие")
    return {"X": raw, "y": y, "E": E, "g": g, "groups": group,
            "conditions": conditions, "nu": normalize_nuisance(E, g),
            "bins": nuisance_bins(E, g), "manifest": manifest}

def split_condition_groups(gids, count, conditions, seed):
    strata = nuisance_bins(conditions[gids, 0], conditions[gids, 1])
    counts = np.unique(strata, return_counts=True)[1]
    use_strata = strata if min(count, len(gids) - count) >= len(counts) and counts.min() >= 2 else None
    return train_test_split(gids, test_size=count, random_state=seed, stratify=use_strata)

DATA = load_dictionaries(DATA_DIR, None if MODE == "train" else CHECKPOINT["data_manifest"])
IDX = None
if DATA is not None:
    source_hashes = [row["sha256"] for row in CHECKPOINT["data_manifest"]]
    current_hashes = [row["sha256"] for row in DATA["manifest"]]
    same_order = current_hashes == source_hashes
    if MODE == "pretrained":
        IDX = {key: value.cpu().numpy() for key, value in CHECKPOINT["source_split_indices"].items()}
    elif MODE == "recalibrate":
        saved = {key: value.cpu().numpy() for key, value in CHECKPOINT["source_split_indices"].items()}
        if {"s_cal", "select", "s_guard"} <= set(saved):
            IDX = {"train": saved["train"], "val": saved["val"], "test": saved["test"],
                   "w_cal": np.sort(np.concatenate([saved["w_cal"], saved["s_cal"], saved["select"]])),
                   "guard": np.sort(np.concatenate([saved["guard"], saved["s_guard"]]))}
        else:
            IDX = saved
    else:
        remaining = np.arange(len(DATA["conditions"]))
        roles = {}
        for offset, (name, fraction) in enumerate([("test", .10), ("guard", .19), ("w_cal", .17), ("val", .04)]):
            count = int(round(fraction * len(DATA["conditions"])))
            if count < 1 or count >= len(remaining):
                raise ValueError("Недостаточно nuisance-условий для независимых частей")
            remaining, chosen = split_condition_groups(remaining, count, DATA["conditions"], SEED + offset)
            roles[name] = chosen
        roles["train"] = remaining
        IDX = {key: np.flatnonzero(np.isin(DATA["groups"], groups)) for key, groups in roles.items()}
    all_ids = np.concatenate(list(IDX.values()))
    if len(all_ids) != len(DATA["y"]) or len(np.unique(all_ids)) != len(DATA["y"]):
        raise ValueError("Части данных должны быть непересекающимися и покрывать все события")
    assigned_groups = [np.unique(DATA["groups"][ids]) for ids in IDX.values()]
    if sum(map(len, assigned_groups)) != len(np.unique(np.concatenate(assigned_groups))):
        raise ValueError("Нельзя распределять повторы одного (E,g) между разными частями")
    split_frame = pd.DataFrame([{"role": key, "events": len(ids),
        "conditions": len(np.unique(DATA["groups"][ids]))} for key, ids in IDX.items()])
    display(split_frame)
    split_frame.to_csv(OUTPUT_DIR / "split.csv", index=False)
    np.savez_compressed(OUTPUT_DIR / "split_indices.npz", **IDX)
    print({"physical_events": len(DATA["y"]), "new_simulations": 0, "remaining_budget": MAX_BUDGET - len(DATA["y"])})
elif MODE != "pretrained":
    raise FileNotFoundError(f"Для MODE={MODE!r} нужны calo_dictionary*.npz в {DATA_DIR}")


In [ ]:
def membership_metrics(y, membership):
    y = np.asarray(y, dtype=np.int64)
    m = np.asarray(membership, dtype=bool)
    if len(y) == 0 or m.shape != (len(y), 2) or not np.isin(y, [0, 1]).all():
        raise ValueError("Нужны непустые y и membership формы (N,2)")
    covered = m[np.arange(len(y)), y]
    powered = ~m[np.arange(len(y)), 1 - y]
    sizes = m.sum(axis=1)
    class_powers = [powered[y == label].mean() for label in (0, 1) if np.any(y == label)]
    result = {"n": len(y), "coverage": float(covered.mean()), "power": float(np.mean(class_powers)),
              "empty": float((sizes == 0).mean()), "ambiguous": float((sizes == 2).mean()),
              "singleton": float((sizes == 1).mean()),
              "correct_singleton": float((covered & (sizes == 1)).mean()), "mean_size": float(sizes.mean())}
    for label in (0, 1):
        mask = y == label
        result[f"coverage_{label}"] = float(covered[mask].mean()) if mask.any() else np.nan
        result[f"power_{label}"] = float(powered[mask].mean()) if mask.any() else np.nan
    return result

def classifier_metrics(y, logits):
    p = np.clip(expit(logits), 1e-15, 1 - 1e-15)
    return {"bce": float(log_loss(y, p, labels=[0, 1])),
            "auc": float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else np.nan,
            "accuracy": float(accuracy_score(y, p >= .5))}

def bernoulli_kl(p, q):
    if p == q:
        return 0.0
    a = 0.0 if p == 0 else p * math.log(p / q)
    b = 0.0 if p == 1 else (1 - p) * math.log((1 - p) / (1 - q))
    return a + b

def kl_upper(k, n, delta):
    if n == 0 or k == n:
        return 1.0
    p = k / n
    target = math.log(1 / delta) / n
    lo, hi = p, 1.0
    for _ in range(60):
        mid = (lo + hi) / 2
        if mid == 1 or bernoulli_kl(p, mid) > target:
            hi = mid
        else:
            lo = mid
    return hi

def holm_adjust(p):
    p = np.asarray(p, dtype=float)
    order = np.argsort(p)
    adjusted = np.minimum(1, np.maximum.accumulate((len(p) - np.arange(len(p))) * p[order]))
    out = np.empty_like(p)
    out[order] = adjusted
    return out

def conditional_audit(y, membership, E, g, groups):
    bin_id = nuisance_bins(E, g)
    results = []
    for family, grouping in (("bins", bin_id), ("exact_conditions", np.asarray(groups))):
        rows = []
        for ia, alpha in enumerate(ALPHAS):
            covered = membership[np.arange(len(y)), ia, y]
            powered = ~membership[np.arange(len(y)), ia, 1 - y]
            for label in (0, 1):
                for group in np.unique(grouping[y == label]):
                    mask = (y == label) & (grouping == group)
                    n, k = int(mask.sum()), int(covered[mask].sum())
                    if n < AUDIT_MIN_N:
                        continue
                    coverage, nominal = k / n, 1 - alpha
                    p_kl = 1.0 if coverage >= nominal else math.exp(-n * bernoulli_kl(coverage, nominal))
                    rows.append({"family": family, "y": label, "group": int(group), "alpha": float(alpha),
                        "n": n, "covered": k, "coverage": coverage, "power": float(powered[mask].mean()),
                        "nominal": nominal, "gap": coverage - nominal,
                        "E_mean": float(np.mean(E[mask])), "g_mean": float(np.mean(g[mask])),
                        "p_binomial": float(binomtest(k, n, nominal, alternative="less").pvalue),
                        "p_chernoff": p_kl})
        if rows:
            frame = pd.DataFrame(rows)
            frame["holm_binomial"] = holm_adjust(frame["p_binomial"])
            frame["holm_chernoff"] = holm_adjust(frame["p_chernoff"])
            results.extend(frame.to_dict("records"))
    return pd.DataFrame(results)


In [ ]:
def fit_preprocessing(raw, train_ids):
    sum_x, sum_x2 = np.zeros(4), np.zeros(4)
    sum_f, sum_f2 = np.zeros(20), np.zeros(20)
    pixels = events = 0
    for start in range(0, len(train_ids), 8192):
        x = raw[train_ids[start:start + 8192]]
        z = np.log1p(x).astype(np.float64)
        f = event_features(x).astype(np.float64)
        sum_x += z.sum(axis=(0, 2, 3))
        sum_x2 += (z * z).sum(axis=(0, 2, 3))
        sum_f += f.sum(axis=0)
        sum_f2 += (f * f).sum(axis=0)
        pixels += len(x) * 64
        events += len(x)
    image_mean = sum_x / pixels
    feature_mean = sum_f / events
    scaler = {"image_mean": image_mean.astype(np.float32),
        "image_std": np.sqrt(np.maximum(sum_x2 / pixels - image_mean**2, 1e-12)).astype(np.float32),
        "feature_mean": feature_mean.astype(np.float32),
        "feature_std": np.maximum(np.sqrt(np.maximum(sum_f2 / events - feature_mean**2, 0)), 1e-6).astype(np.float32)}
    return {key: torch.from_numpy(value) for key, value in scaler.items()}

class IndexDataset(Dataset):
    def __init__(self, indices):
        self.indices = np.asarray(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        return int(self.indices[i])

def make_loader(indices, scaler, seed):
    def collate(batch):
        ids = np.asarray(batch, dtype=np.int64)
        maps, f = preprocess(DATA["X"][ids], scaler)
        return (torch.from_numpy(maps), torch.from_numpy(f),
                torch.from_numpy(DATA["y"][ids]), torch.from_numpy(DATA["nu"][ids]))
    return DataLoader(IndexDataset(indices), batch_size=CONFIG["batch_size"], shuffle=True,
        num_workers=0, collate_fn=collate, pin_memory=DEVICE.type == "cuda",
        generator=torch.Generator().manual_seed(seed))

def train_rescnn(scaler):
    seed_everything(SEED)
    model = CaloResCNN(CONFIG).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
    amp_scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    loader = make_loader(IDX["train"], scaler, SEED)
    best_bce, best_state, stale, history = np.inf, None, 0, []
    for epoch in range(1, MAX_EPOCHS + 1):
        start = time.time()
        model.train()
        bce_sum = aux_sum = 0.0
        for maps, features, labels, nuisance_target in loader:
            maps, features = maps.to(DEVICE), features.to(DEVICE)
            labels, nuisance_target = labels.to(DEVICE), nuisance_target.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                logits, mu, scale = model(maps, features)
                bce = nn.functional.binary_cross_entropy_with_logits(logits, labels.float())
                pos = torch.arange(len(labels), device=DEVICE)
                my, sy = mu[pos, labels], scale[pos, labels]
                auxiliary = (.5 * ((nuisance_target - my) / sy)**2 + torch.log(sy)).mean()
                loss = bce + CONFIG["aux_weight"] * auxiliary
            if not torch.isfinite(loss):
                raise FloatingPointError("Неконечная функция потерь")
            amp_scaler.scale(loss).backward()
            amp_scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            amp_scaler.step(optimizer)
            amp_scaler.update()
            bce_sum += float(bce.detach()) * len(labels)
            aux_sum += float(auxiliary.detach()) * len(labels)
        scheduler.step()
        val_logits = predict_logits(model, DATA["X"], scaler, indices=IDX["val"])
        metrics = classifier_metrics(DATA["y"][IDX["val"]], val_logits)
        row = {"epoch": epoch, "train_bce": bce_sum / len(IDX["train"]),
            "train_aux": aux_sum / len(IDX["train"]), "val_bce": metrics["bce"],
            "val_auc": metrics["auc"], "seconds": time.time() - start}
        history.append(row)
        if metrics["bce"] < best_bce - 1e-5:
            best_bce = metrics["bce"]
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            stale = 0
            torch.save({"config": CONFIG, "state_dict": best_state, "preprocess": scaler,
                        "epoch": epoch}, OUTPUT_DIR / "best_training_state.pt")
        else:
            stale += 1
        if epoch == 1 or epoch % 5 == 0 or stale >= PATIENCE or epoch == MAX_EPOCHS:
            print(f"ResCNN epoch {epoch}: val_BCE={metrics['bce']:.6f}, val_AUC={metrics['auc']:.6f}", flush=True)
        if stale >= PATIENCE:
            break
    if best_state is None:
        raise RuntimeError("Не получено ни одного валидного checkpoint")
    model.load_state_dict(best_state)
    return model.eval(), pd.DataFrame(history), best_state

if MODE == "train":
    SCALER = fit_preprocessing(DATA["X"], IDX["train"])
    MODEL, HISTORY, state = train_rescnn(SCALER)
    TRAIN_PRIOR_1 = float(DATA["y"][IDX["train"]].mean())
    CHECKPOINT = {"format_version": 1, "model_name": "ResCNN", "config": CONFIG,
        "seed": SEED, "state_dict": state, "train_prior_1": TRAIN_PRIOR_1,
        "preprocess": SCALER, "data_manifest": DATA["manifest"],
        "source_split_indices": {key: torch.from_numpy(value.copy()) for key, value in IDX.items()},
        "training_history": HISTORY.to_dict("records"), "saved_test_metrics": [],
        "saved_classifier_metrics": [], "saved_audit_summary": [], "naps": None,
        "budget": {"physical_used": len(DATA["y"]), "new_physical_simulations": 0},
        "provenance": {"optimizer": "Adam", "weights_retrained_during_export": True}}
    HISTORY.to_csv(OUTPUT_DIR / "training_history.csv", index=False)
else:
    HISTORY = pd.DataFrame(CHECKPOINT["training_history"])
display(HISTORY.tail())


In [ ]:
def compressed_cdf_rows(ids, stats, label, thresholds):
    blocks, targets, weights = [], [], []
    for gid in np.unique(DATA["groups"][ids]):
        mask = (DATA["groups"][ids] == gid) & (DATA["y"][ids] == label)
        if not mask.any():
            continue
        scores = np.sort(stats[mask, label])
        counts = np.searchsorted(scores, thresholds, side="right")
        nu = DATA["nu"][ids[np.flatnonzero(mask)[0]]]
        rows = np.column_stack([np.tile(nu, (len(thresholds), 1)), thresholds]).astype(np.float32)
        for target, weight in ((0, len(scores) - counts), (1, counts)):
            keep = weight > 0
            if keep.any():
                endpoint_weight = np.where((thresholds[keep] == thresholds[0]) | (thresholds[keep] == thresholds[-1]), 8.0, 1.0)
                blocks.append(rows[keep])
                targets.append(np.full(keep.sum(), target))
                weights.append(weight[keep] * endpoint_weight)
    return np.concatenate(blocks), np.concatenate(targets), np.concatenate(weights)

def fit_rejection_probabilities(logits):
    from catboost import CatBoostClassifier, Pool
    ids = IDX["w_cal"]
    groups = np.unique(DATA["groups"][ids])
    fit_groups, val_groups = split_condition_groups(groups, max(1, len(groups) // 5), DATA["conditions"], SEED + 200)
    fit_mask = np.isin(DATA["groups"][ids], fit_groups)
    stats = statistics(logits, TRAIN_PRIOR_1)
    models, bounds, rows = {}, {}, []
    for label in (0, 1):
        true_scores = stats[fit_mask & (DATA["y"][ids] == label), label]
        levels = np.unique(np.r_[np.linspace(.001, .25, 40), np.linspace(.3, .99, 15)])
        thresholds = np.unique(np.r_[np.quantile(true_scores, levels),
                                     np.linspace(true_scores.min() - 2, true_scores.max() + 2, 16)].astype(np.float32))
        fx, fy, fw = compressed_cdf_rows(ids[fit_mask], stats[fit_mask], label, thresholds)
        vx, vy, vw = compressed_cdf_rows(ids[~fit_mask], stats[~fit_mask], label, thresholds)
        model = CatBoostClassifier(iterations=W_ITERATIONS, depth=4, l2_leaf_reg=10,
            learning_rate=.05, border_count=BORDER_COUNT, loss_function="Logloss",
            monotone_constraints=[0, 0, 1], random_seed=SEED + label,
            thread_count=THREADS, verbose=False, allow_writing_files=False)
        model.fit(Pool(fx, fy, weight=fw), eval_set=Pool(vx, vy, weight=vw),
                  early_stopping_rounds=70, use_best_model=True)
        models[label] = model
        bounds[label] = (float(true_scores.min() - 20), float(true_scores.max() + 20))
        rows.append({"y": label, "trees": model.tree_count_,
                     "validation_logloss": float(log_loss(vy, model.predict_proba(vx)[:, 1], sample_weight=vw))})
        model.save_model(str(OUTPUT_DIR / f"W_{label}.cbm"))
    return models, bounds, pd.DataFrame(rows)

def rejection_probability(model, nuisance, threshold):
    points = np.asarray(nuisance, dtype=np.float32).reshape(-1, 2)
    thresholds = np.broadcast_to(threshold, (len(points),))
    return model.predict_proba(np.column_stack([points, thresholds]).astype(np.float32))[:, 1]

def inverse_rejection_probability(model, points, probability, lower, upper, iterations=32):
    points = np.asarray(points, dtype=np.float32)
    if not 0 < probability < 1:
        raise ValueError("Уровень вероятности должен лежать между 0 и 1")
    lo, hi = np.full(len(points), lower), np.full(len(points), upper)
    valid = (rejection_probability(model, points, lo) <= probability) & (rejection_probability(model, points, hi) > probability)
    for _ in range(iterations):
        mid = (lo + hi) / 2
        safe = rejection_probability(model, points, mid) <= probability
        lo = np.where(safe, mid, lo)
        hi = np.where(safe, hi, mid)
    cutoff = np.nextafter(lo.astype(np.float32), np.float32(-np.inf)).astype(np.float64)
    valid &= rejection_probability(model, points, cutoff) <= probability
    return np.where(valid, cutoff, -np.inf)

def nuisance_partition(model):
    axes = []
    for feature in (0, 1):
        borders = np.unique(np.asarray(model.get_borders().get(feature, []), dtype=np.float32))
        borders = borders[(borders >= 0) & (borders < 1)]
        hi = np.r_[borders, np.float32(1)].astype(np.float32)
        valid_lo = np.r_[np.float32(0), np.nextafter(borders, np.float32(np.inf))]
        outer_lo = np.r_[np.float32(0), borders]
        keep = valid_lo <= hi
        axes.append((outer_lo[keep], hi[keep], valid_lo[keep]))
    i, j = np.meshgrid(np.arange(len(axes[0][0])), np.arange(len(axes[1][0])), indexing="ij")
    lo = np.column_stack([axes[0][0][i.ravel()], axes[1][0][j.ravel()]])
    hi = np.column_stack([axes[0][1][i.ravel()], axes[1][1][j.ravel()]])
    representative = np.column_stack([axes[0][2][i.ravel()], axes[1][2][j.ravel()]]).astype(np.float32)
    if len(representative) > 100_000:
        raise RuntimeError("Слишком много nuisance-ячеек")
    return lo, hi, representative

def worst_case_cutoffs(models, bounds):
    raw, worst, basis = np.empty((len(ALPHAS), 2)), np.empty((len(ALPHAS), 2, 2, 2)), {}
    for label in (0, 1):
        lo, hi, points = nuisance_partition(models[label])
        cuts = np.column_stack([inverse_rejection_probability(models[label], points,
            float(alpha), *bounds[label]) for alpha in ALPHAS])
        if not np.all(cuts[:, 1:] >= cuts[:, :-1]):
            raise RuntimeError("Обратная rejection probability должна возрастать по alpha")
        raw[:, label] = cuts.min(axis=0)
        for ia in range(len(ALPHAS)):
            index = int(cuts[:, ia].argmin())
            worst[ia, label] = np.stack([lo[index], hi[index]])
        basis[label] = {"lo": lo, "hi": hi, "cutoffs": cuts}
        np.savez_compressed(OUTPUT_DIR / f"nuisance_cells_{label}.npz", **basis[label])
    return raw, worst, basis


In [ ]:
def calibrate_margins(logits, raw_cutoffs, ids):
    stats = statistics(logits, TRAIN_PRIOR_1)
    family = 2 * N_BINS * len(ALPHAS) * (len(MARGINS) - 1)
    delta = GUARD_DELTA / family
    margins = np.full((len(ALPHAS), 2), np.inf)
    rows = []
    for ia, alpha in enumerate(ALPHAS):
        for label in (0, 1):
            for margin in MARGINS[:-1]:
                rejected = stats[:, label] <= raw_cutoffs[ia, label] - margin
                upper = 0.0
                for bin_id in range(N_BINS):
                    mask = (DATA["y"][ids] == label) & (DATA["bins"][ids] == bin_id)
                    bound = 0.0 if np.isneginf(raw_cutoffs[ia, label]) else kl_upper(int(rejected[mask].sum()), int(mask.sum()), delta)
                    upper = max(upper, bound)
                rows.append({"alpha": float(alpha), "y": label, "margin": float(margin), "max_bin_ucb": upper})
                if upper <= alpha and np.isinf(margins[ia, label]):
                    margins[ia, label] = margin
    return margins, pd.DataFrame(rows)

if MODE in {"recalibrate", "train"}:
    if importlib.util.find_spec("catboost") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "catboost==1.2.8"])
    W_LOGITS = predict_logits(MODEL, DATA["X"], SCALER, indices=IDX["w_cal"])
    W_MODELS, W_BOUNDS, W_VALIDATION = fit_rejection_probabilities(W_LOGITS)
    display(W_VALIDATION)
    W_VALIDATION.to_csv(OUTPUT_DIR / "W_validation.csv", index=False)
    RAW_CUTOFFS, WORST_CELLS, NUISANCE_BASIS = worst_case_cutoffs(W_MODELS, W_BOUNDS)
    GUARD_LOGITS = predict_logits(MODEL, DATA["X"], SCALER, indices=IDX["guard"])
    CALIBRATION_MARGINS, GUARD_TABLE = calibrate_margins(GUARD_LOGITS, RAW_CUTOFFS, IDX["guard"])
    GUARD_TABLE.to_csv(OUTPUT_DIR / "guard.csv", index=False)
    adjusted = RAW_CUTOFFS - CALIBRATION_MARGINS
    FINAL_THRESHOLDS = np.minimum.accumulate(adjusted[::-1], axis=0)[::-1].copy()
    CHECKPOINT["naps"] = {"alphas": torch.from_numpy(ALPHAS.copy()), "gamma": RECALIBRATION_GAMMA,
        "confidence_set": "full_nuisance_domain", "nuisance_domain": {"E": [1.0, 100.0], "g": [.7, 1.3]},
        "raw_cutoffs": torch.from_numpy(RAW_CUTOFFS), "margins": torch.from_numpy(CALIBRATION_MARGINS),
        "thresholds": torch.from_numpy(FINAL_THRESHOLDS), "worst_cells_normalized": torch.from_numpy(WORST_CELLS),
        "continuum_certified": False, "coverage_scope": "bin-average under current simulation design",
        "calibration_source": "new gamma=0 rejection probability + independent bin guard"}
    if MODE == "recalibrate":
        CHECKPOINT["source_split_indices"] = {key: torch.from_numpy(value.copy()) for key, value in IDX.items()}
    CHECKPOINT["saved_test_metrics"] = []
    CHECKPOINT["saved_classifier_metrics"] = []
    CHECKPOINT["saved_audit_summary"] = []
    torch.save(CHECKPOINT, OUTPUT_DIR / "NAPS_Calo_ResCNN_gamma0.pt")

PREDICTOR = ResCNNNAPSPredictor(CHECKPOINT, device=str(DEVICE))
threshold_rows = []
worst = CHECKPOINT["naps"]["worst_cells_normalized"].cpu().numpy()
for ia, alpha in enumerate(ALPHAS):
    for label in (0, 1):
        cell = worst[ia, label]
        threshold_rows.append({"alpha": float(alpha), "y": label,
            "raw_cutoff": float(CHECKPOINT["naps"]["raw_cutoffs"][ia, label]),
            "margin": float(CHECKPOINT["naps"]["margins"][ia, label]),
            "final_cutoff": float(PREDICTOR.thresholds[ia, label]),
            "E_low": float(100**cell[0, 0]), "E_high": float(100**cell[1, 0]),
            "g_low": float(.7 + .6 * cell[0, 1]), "g_high": float(.7 + .6 * cell[1, 1])})
THRESHOLDS_TABLE = pd.DataFrame(threshold_rows)
display(THRESHOLDS_TABLE)
THRESHOLDS_TABLE.to_csv(OUTPUT_DIR / "thresholds.csv", index=False)
print({"active_gamma": CHECKPOINT["naps"]["gamma"],
    "confidence_set": CHECKPOINT["naps"]["confidence_set"],
    "continuum_certified": CHECKPOINT["naps"]["continuum_certified"],
    "coverage_scope": CHECKPOINT["naps"]["coverage_scope"]})


In [ ]:
TEST_METRICS = pd.DataFrame()
AUDIT = pd.DataFrame()
AUDIT_SUMMARY = pd.DataFrame()
if DATA is not None:
    TEST_IDS = IDX["test"]
    TEST_Y = DATA["y"][TEST_IDS]
    TEST_LOGITS = predict_logits(MODEL, DATA["X"], SCALER, indices=TEST_IDS)
    TEST_MEMBERSHIP = statistics(TEST_LOGITS, TRAIN_PRIOR_1)[:, None, :] > PREDICTOR.thresholds[None, :, :]
    rows = []
    for scenario, mask in (("Reference", np.ones(len(TEST_IDS), dtype=bool)),
                           ("Low energy E<=5", DATA["E"][TEST_IDS] <= 5)):
        if not mask.any():
            continue
        for ia, alpha in enumerate(ALPHAS):
            rows.append({"evaluation_source": "current_held_out_data", "scenario": scenario,
                         "alpha": float(alpha), **membership_metrics(TEST_Y[mask], TEST_MEMBERSHIP[mask, ia])})
    TEST_METRICS = pd.DataFrame(rows)
    CLASSIFIER_METRICS = pd.DataFrame([{"evaluation_source": "current_held_out_data",
                                      **classifier_metrics(TEST_Y, TEST_LOGITS)}])
    AUDIT = conditional_audit(TEST_Y, TEST_MEMBERSHIP, DATA["E"][TEST_IDS],
                              DATA["g"][TEST_IDS], DATA["groups"][TEST_IDS])
    if not AUDIT.empty:
        AUDIT_SUMMARY = AUDIT.groupby(["family", "alpha"]).agg(
            min_coverage=("coverage", "min"), min_gap=("gap", "min"),
            binomial_failures=("holm_binomial", lambda p: int((p < AUDIT_SIGNIFICANCE).sum())),
            chernoff_failures=("holm_chernoff", lambda p: int((p < AUDIT_SIGNIFICANCE).sum()))).reset_index()
    sample_ids = TEST_IDS[:min(512, len(TEST_IDS))]
    p, codes = PREDICTOR.predict(DATA["X"][sample_ids])
    expected_codes = encode_membership(TEST_MEMBERSHIP[:len(sample_ids)])
    if not np.array_equal(codes, expected_codes):
        raise AssertionError("Загруженный predictor отличается от оценённого pipeline")
    if not np.allclose(p, expit(TEST_LOGITS[:len(sample_ids)]), rtol=1e-5, atol=1e-6):
        raise AssertionError("Вероятности checkpoint отличаются от test pipeline")
    PREDICTOR.export(DATA["X"][sample_ids], OUTPUT_DIR / "example_submission.csv", event_ids=sample_ids)
else:
    TEST_METRICS = pd.DataFrame(CHECKPOINT["saved_test_metrics"])
    if not TEST_METRICS.empty:
        TEST_METRICS["evaluation_source"] = "saved_original_test"
        TEST_METRICS["singleton"] = 1 - TEST_METRICS["ambiguous"] - TEST_METRICS["empty"]
        TEST_METRICS = TEST_METRICS.drop(columns=["pipeline"], errors="ignore")
    CLASSIFIER_METRICS = pd.DataFrame(CHECKPOINT["saved_classifier_metrics"])
    if not CLASSIFIER_METRICS.empty:
        CLASSIFIER_METRICS["evaluation_source"] = "saved_original_test"
    AUDIT_SUMMARY = pd.DataFrame(CHECKPOINT["saved_audit_summary"]).drop(columns=["pipeline"], errors="ignore")
    if not AUDIT_SUMMARY.empty:
        AUDIT_SUMMARY["evaluation_source"] = "saved_original_test"
display(CLASSIFIER_METRICS)
display(TEST_METRICS.round(6))
display(AUDIT_SUMMARY.round(6))
TEST_METRICS.to_csv(OUTPUT_DIR / "test_metrics.csv", index=False)
CLASSIFIER_METRICS.to_csv(OUTPUT_DIR / "classifier_metrics.csv", index=False)
AUDIT.to_csv(OUTPUT_DIR / "conditional_coverage.csv", index=False)
AUDIT_SUMMARY.to_csv(OUTPUT_DIR / "audit_summary.csv", index=False)
if not TEST_METRICS.empty:
    reference = TEST_METRICS.loc[TEST_METRICS["scenario"] == "Reference"]
    display(pd.DataFrame([{"mean_power_over_alphas": float(reference["power"].mean()),
        "evaluation_source": reference["evaluation_source"].iloc[0]}]))


In [ ]:
if not TEST_METRICS.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for scenario, frame in TEST_METRICS.groupby("scenario", sort=False):
        frame = frame.sort_values("alpha")
        axes[0].plot(frame["alpha"], frame["coverage"], "o-", label=scenario)
        axes[1].plot(frame["alpha"], frame["power"], "o-", label=scenario)
    axes[0].plot(ALPHAS, 1 - ALPHAS, "k--", label="1-alpha")
    ref = TEST_METRICS[TEST_METRICS["scenario"] == "Reference"].sort_values("alpha")
    axes[2].plot(ref["alpha"], ref["ambiguous"], "o-", label="ambiguous")
    axes[2].plot(ref["alpha"], ref["singleton"], "s-", label="singleton")
    axes[2].plot(ref["alpha"], ref["empty"], "^-", label="empty")
    for ax, title in zip(axes, ["Coverage", "Balanced power", "Prediction sets"]):
        ax.set(xlabel="alpha", title=title, ylim=(0, 1.02))
        ax.set_xticks(ALPHAS)
        ax.grid(alpha=.3)
        ax.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "figures/coverage_power.png", dpi=160)
    plt.show()

if not HISTORY.empty:
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(HISTORY["epoch"], HISTORY["train_bce"], label="train BCE")
    ax.plot(HISTORY["epoch"], HISTORY["val_bce"], label="validation BCE")
    ax.set(xlabel="epoch", ylabel="BCE", title="ResCNN")
    ax.grid(alpha=.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "figures/training.png", dpi=160)
    plt.show()

if not AUDIT.empty:
    for metric in ("coverage", "power"):
        frame = AUDIT[AUDIT["family"] == "bins"].copy()
        frame["E_bin"] = frame["group"] // 4
        frame["g_bin"] = frame["group"] % 4
        fig, axes = plt.subplots(2, 3, figsize=(14, 7), constrained_layout=True)
        for label in (0, 1):
            for ia, alpha in enumerate(ALPHAS):
                part = frame[(frame["y"] == label) & (frame["alpha"] == alpha)]
                matrix = part.pivot(index="E_bin", columns="g_bin", values=metric).reindex(index=range(6), columns=range(4))
                image = axes[label, ia].imshow(matrix.to_numpy(), origin="lower", vmin=0, vmax=1, aspect="auto", cmap="viridis")
                axes[label, ia].set(title=f"{metric}: y={label}, alpha={alpha}", xlabel="g bin", ylabel="E bin")
                axes[label, ia].set_xticks(range(4), [f"{G_EDGES[j]:.2f}-{G_EDGES[j+1]:.2f}" for j in range(4)], rotation=45)
                axes[label, ia].set_yticks(range(6), [f"{E_EDGES[j]:.1f}-{E_EDGES[j+1]:.1f}" for j in range(6)])
        fig.colorbar(image, ax=axes.ravel().tolist(), shrink=.8)
        fig.savefig(OUTPUT_DIR / f"figures/conditional_{metric}.png", dpi=160)
        plt.show()


In [ ]:
SESSION_OUTPUTS = []
for path in sorted(DATA_DIR.glob(SESSION_PATTERN)):
    with np.load(path, allow_pickle=False) as data:
        key = "X" if "X" in data.files else "values" if "values" in data.files else None
        if key is None:
            raise ValueError(f"В {path.name} нет наблюдений X или values")
        observations = np.asarray(data[key], dtype=np.float32)
        if observations.shape == (4, 8, 8):
            observations = observations[None]
        event_ids = np.asarray(data["event_id"]) if "event_id" in data.files else np.arange(len(observations))
    destination = OUTPUT_DIR / f"submission_{path.stem}.csv"
    frame = PREDICTOR.export(observations, destination, event_ids=event_ids)
    SESSION_OUTPUTS.append({"session": path.name, "events": len(frame), "submission": str(destination)})
    display(frame.head())
display(pd.DataFrame(SESSION_OUTPUTS))
if DOWNLOAD_RESULTS and IN_COLAB:
    from google.colab import files
    archive_path = ROOT_DIR / "NAPS_ResCNN_results.zip"
    with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in OUTPUT_DIR.rglob("*"):
            if path.is_file():
                archive.write(path, path.relative_to(OUTPUT_DIR))
    files.download(str(archive_path))
